# Passive LC Decoderless State Output

This notebook runs the strict no-decoder passive LC toy experiment. The LC final state is mapped directly to CIFAR pixels with `tanh(final_state)`.

For CIFAR-10, `3 * 32 * 32 = 3072` pixel values. The LC state is `concat(phi, V)`, so `state_dim = 2 * n_oscillators`. This notebook uses `n_oscillators = 1536` so the LC state exactly matches the flattened image dimension.

## Setup

In [ ]:
!git clone --branch passive-lc-toy https://github.com/joe-singh/Nonlinear-LC-Images.git
%cd Nonlinear-LC-Images
!pip install -q -e . --no-deps
!pip install -q torchdiffeq torchvision tqdm clean-fid

## Decoderless LC Run

In [ ]:
!python scripts/train_passive_lc_toy_cifar10.py \
    --device cuda \
    --precision bf16 \
    --seed 42 \
    --decoder-type state \
    --n-oscillators 1536 \
    --topology ring \
    --num-steps 8 \
    --method rk4 \
    --epochs 25 \
    --batch-size 128 \
    --subset-size 10000 \
    --out-dir runs/state_decoderless_ring_n1536

## Inspect Samples

In [ ]:
from IPython.display import Image
from IPython.display import display as show

show(Image("runs/state_decoderless_ring_n1536/samples_epoch_000.png"))
show(Image("runs/state_decoderless_ring_n1536/samples_epoch_025.png"))

## FID

In [ ]:
!python scripts/eval_passive_lc_toy_fid.py \
    --checkpoint runs/state_decoderless_ring_n1536/final.pt \
    --num-samples 5000 \
    --batch-size 256 \
    --device cuda \
    --output runs/state_decoderless_ring_n1536/fid_5k.json

## Diagnostics

In [ ]:
import json
from pathlib import Path

run_dir = Path("runs/state_decoderless_ring_n1536")
with (run_dir / "diagnostics.json").open() as f:
    diagnostics = json.load(f)
with (run_dir / "fid_5k.json").open() as f:
    fid = json.load(f)

last = diagnostics["epochs"][-1]
print("FID:", fid["fid"])
print("optimizer_lrs:", diagnostics["optimizer_lrs"])
print("grad norms:", last["mean_grad_norm"])
print("LC delta:", last["lc_parameter_delta_from_init"])